# Generate label/document embeddings (Fine-tuning)

고려대학교 보건과학대학 바이오의공학부

2021250031 정예준

In [ ]:
import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))

cuda available: True
device name: Tesla T4


In [ ]:
# Import libraries
import numpy as np
import torch
import random
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
from transformers import (
    BertTokenizerFast,
    BertForMaskedLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [ ]:
# Random seed
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

In [ ]:
# Root dataset directory
ROOT = Path("Amazon_products")

# Corpus paths
TRAIN_CORPUS_PATH = ROOT / "train" / "train_corpus.txt"
TEST_CORPUS_PATH = ROOT / "test" / "test_corpus.txt"

# Class-related information
CLASS_HIERARCHY_PATH = ROOT / "class_hierarchy.txt"
CLASS_KEYWORDS_PATH = ROOT / "class_related_keywords.txt"
CLASS_NAMES_PATH = ROOT / "classes.txt"

In [ ]:
# Data loading function
def load_corpus(path):
    """
    Load corpus file (train/test).
    Each line: `<int_id> <space> <review text...>`
    Returns: {doc_id: text}
    """
    corpus = {}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            doc_id_str, text = line.split(maxsplit=1)
            doc_id = int(doc_id_str)
            corpus[doc_id] = text.strip()
    return corpus

def load_class_names(path):
    """
    Load class name and id.
    Each line: `<id> <class_name>`
    Returns:
      - class_names: index == class_id (list)
      - name_to_id : class_name -> class_id (dictionary)
    """
    class_names = []
    name_to_id = {}

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            cls_id = int(parts[0])
            cls_name = parts[1]

            class_names.append(cls_name)
            name_to_id[cls_name] = cls_id

    return class_names, name_to_id

def load_class_hierarchy(path):
    """
    Load class hierarchy edges.
    Each line: `<parent_id> <child_id>`
    Returns:
      - parent_to_children: {parent_id: [child_id, ...]}
      - child_to_parents: {child_id: [parent_id, ...]}
    """
    parent_to_children = defaultdict(list)
    child_to_parents = defaultdict(list)

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parent_str, child_str = line.split()
            parent = int(parent_str)
            child = int(child_str)
            parent_to_children[parent].append(child)
            child_to_parents[child].append(parent)

    return dict(parent_to_children), dict(child_to_parents)

def load_class_keywords(path, class_names):
    """
    Load class-related keywords.
    Each line: `<class_name>:kw1,kw2,...`
    Returns: {class_id: [kw1, kw2, ...]}
    """
    name_to_id = {name: idx for idx, name in enumerate(class_names)}
    class_keywords = {}

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cls_name, kws_str = line.split(":", maxsplit=1)
            cls_name = cls_name.strip()
            kws = [k.strip() for k in kws_str.split(",") if k.strip()]
            cls_id = name_to_id[cls_name]
            class_keywords[cls_id] = kws

    return class_keywords

In [ ]:
# Load data
train_corpus = load_corpus(TRAIN_CORPUS_PATH)
test_corpus  = load_corpus(TEST_CORPUS_PATH)
class_names, name_to_id = load_class_names(CLASS_NAMES_PATH)
parent2children, child2parents = load_class_hierarchy(CLASS_HIERARCHY_PATH)
class_keywords = load_class_keywords(CLASS_KEYWORDS_PATH, class_names)

In [ ]:
# Make label texts for BERT
def build_label_texts(class_names, class_keywords):
    """
    Make label texts.
    ex) 'grocery gourmet food snacks condiments ...'
    """
    label_texts = []

    for cid, name in enumerate(class_names):
        pretty_name = name.replace("_", " ")
        keywords = class_keywords.get(cid, [])
        text = " ".join([pretty_name] + keywords)
        label_texts.append(text)

    return label_texts

In [ ]:
# MLM dataset for Fine-tuning
class MLMDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.texts = list(texts)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        item = {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
        }
        return item

In [ ]:
# MLM fine-tuning (Domain-Adaptive Pretraining)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Prepare tokenizer and MLM model (based on pretrained BERT)
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
mlm_model = BertForMaskedLM.from_pretrained("bert-base-uncased")
mlm_model.to(device)

# Prepare texts
train_texts = list(train_corpus.values())
test_texts = list(test_corpus.values())
label_texts = build_label_texts(class_names, class_keywords)

all_texts = train_texts + test_texts + label_texts
print(f"Total texts for MLM fine-tuning: {len(all_texts)}")

# Dataset / DataLoader / Collator
mlm_dataset = MLMDataset(all_texts, tokenizer, max_length=128)

mlm_batch_size = 32
mlm_dataloader = DataLoader(
    mlm_dataset,
    batch_size=mlm_batch_size,
    shuffle=True,
    collate_fn=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15,
    ),
)

# Optimizer / Scheduler
mlm_epochs = 2
mlm_lr = 2e-5

optimizer = optim.AdamW(mlm_model.parameters(), lr=mlm_lr)

num_training_steps = mlm_epochs * len(mlm_dataloader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

# MLM training loop
mlm_model.train()
global_step = 0

for epoch in range(1, mlm_epochs + 1):
    total_loss = 0.0
    for batch in tqdm(mlm_dataloader, desc=f"MLM Epoch {epoch}"):
        # batch: {'input_ids', 'attention_mask', 'labels'}
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = mlm_model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        global_step += 1

    avg_loss = total_loss / max(1, len(mlm_dataloader))
    print(f"[MLM Epoch {epoch}] Avg loss: {avg_loss:.4f}")

# Save fine-tuned BERT
DAPT_DIR = ROOT / "bert_dapt_mlm"
DAPT_DIR.mkdir(exist_ok=True)

mlm_model.save_pretrained(DAPT_DIR)
tokenizer.save_pretrained(DAPT_DIR)

In [ ]:
# Prepare fine-tuned BERT for embedding

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = Path("Amazon_products")
DAPT_DIR = ROOT / "bert_dapt_mlm"

tokenizer = BertTokenizerFast.from_pretrained(DAPT_DIR)
mlm_model = BertForMaskedLM.from_pretrained(DAPT_DIR)
mlm_model.to(device)
mlm_model.eval()

model = mlm_model.bert
model.to(device)
model.eval()

print("Loaded fine-tuned BERT from:", DAPT_DIR)

Loaded fine-tuned BERT from: Amazon_products/bert_dapt_mlm


In [ ]:
def mean_pooling(model_output, attention_mask):
    """
    Apply mean pooling on BERT token embeddings, masking out padding tokens.
    Args:
        model_output: Output object from a BERT model (contains last_hidden_state).
        attention_mask (torch.Tensor): Attention mask of shape (batch_size, seq_len),
                                       where 1 = real token and 0 = padding.
    Returns:
        torch.Tensor: Sentence embeddings of shape (batch_size, hidden_size).
    """
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask

def encode_texts(texts, batch_size=64):
    """
    Encode a list of texts into mean-pooled BERT embeddings.
    Args:
        texts (list of str): Input texts to encode.
        batch_size (int, optional): Batch size for encoding. Default is 64.
    Returns:
        torch.Tensor: Tensor of shape (len(texts), hidden_size) containing embeddings.
    """
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            output = model(**encoded)

        embeddings = mean_pooling(output, encoded["attention_mask"])
        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)

In [ ]:
# Initial label embeddings
ROOT = Path("Amazon_products")
LABEL_EMB_PATH = ROOT / "label_bert_mean_dapt.pt"

label_texts = build_label_texts(class_names, class_keywords)
label_init_emb = encode_texts(label_texts)
torch.save({"embeddings": label_init_emb}, LABEL_EMB_PATH)

100%|██████████| 9/9 [00:02<00:00,  4.34it/s]


In [ ]:
# Train corpus embeddings
ROOT = Path("Amazon_products")
TRAIN_EMB_PATH = ROOT / "train_bert_mean_dapt.pt"

train_ids = sorted(train_corpus.keys())
train_texts = [train_corpus[i] for i in train_ids]
train_doc_embs = encode_texts(train_texts)
torch.save({"ids": train_ids, "embeddings": train_doc_embs}, TRAIN_EMB_PATH)

100%|██████████| 461/461 [11:52<00:00,  1.55s/it]


In [ ]:
# Test corpus embeddings
ROOT = Path("Amazon_products")
TEST_EMB_PATH  = ROOT / "test_bert_mean_dapt.pt"

test_ids = sorted(test_corpus.keys())
test_texts = [test_corpus[i] for i in test_ids]
test_doc_embs = encode_texts(test_texts)
torch.save({"ids": test_ids, "embeddings": test_doc_embs}, TEST_EMB_PATH)

100%|██████████| 308/308 [07:57<00:00,  1.55s/it]
